<a href="https://colab.research.google.com/github/jenitchadha/SDS-510/blob/main/MOD_4_BASICS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Projects & Quizzes

GOALS FOR THIS PROJECT:
1. Scrape ASU news website.
2. look for news articles
3. get keywords for each articles
4. save keyword and article url in a csv file.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
#LOADING THE ASU HOMEPAGE
url = "https://news.asu.edu/"
response = requests.get(url)
print(response.status_code)

200


In [4]:
# Uaing BeautifulSoup so I can find links and article text.
soup = BeautifulSoup(response.text, "html.parser")
print(soup)

<!DOCTYPE html>

<html dir="ltr" lang="en" prefix="og: https://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<script>window.dataLayer = window.dataLayer || [];</script>
<script>
    // Google Tag Manager - GTM-KDWN8Z
    (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
    new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
    j=d.createElement(s),dl=l!='dataLayer'?'&amp;l='+l:'';j.async=true;j.src=
    '//www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })(window,document,'script','dataLayer','GTM-KDWN8Z');
</script>
<link href="https://news.asu.edu/" rel="shortlink"/>
<link href="https://news.asu.edu/" rel="canonical"/>
<meta content="657836F28A73675EDA691E52E47A0DB9" name="msvalidate.01"/>
<meta content="Drupal 10 (https://www.drupal.org)" name="Generator"/>
<meta content="width" name="MobileOptimized"/>
<meta content="true" name="HandheldFriendly"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<m

In [10]:
# finding all <a> elements and
#will get the total number of links found.

links = soup.find_all("a", href=True)
print("Total links found:", len(links))
for link in links[:10]: #this will help in getting top n rows
    print("Text:", link.get_text(" ", strip=True))
    print("URL:", link["href"]) #href is a part of an html that stores its desgignated address
    print()

Total links found: 243
Text: Skip to main content
URL: #skip-to-content

Text: Report an accessibility problem
URL: https://accessibility.asu.edu/report

Text: ASU Home
URL: https://asu.edu

Text: My ASU
URL: https://my.asu.edu

Text: Colleges and Schools
URL: https://www.asu.edu/academics/colleges-schools

Text: Sign In
URL: /caslogin

Text: 
URL: https://www.asu.edu

Text: ASU News
URL: https://news.asu.edu/

Text: Subscribe
URL: /subscribe

Text: Search ASU News Stories
URL: /search



In [11]:
for link in links:
    text = link.get_text(" ", strip=True)
    address = link["href"]
    print("Text:", text)
    print("URL:", address)
    print()
    # this will help me get the link's text and address

Text: Skip to main content
URL: #skip-to-content

Text: Report an accessibility problem
URL: https://accessibility.asu.edu/report

Text: ASU Home
URL: https://asu.edu

Text: My ASU
URL: https://my.asu.edu

Text: Colleges and Schools
URL: https://www.asu.edu/academics/colleges-schools

Text: Sign In
URL: /caslogin

Text: 
URL: https://www.asu.edu

Text: ASU News
URL: https://news.asu.edu/

Text: Subscribe
URL: /subscribe

Text: Search ASU News Stories
URL: /search

Text: Home
URL: /

Text: Browse
URL: 

Text: Browse Stories by Topic
URL: 

Text: Science and Technology
URL: /saf/science-and-technology

Text: Arts, Humanities and Education
URL: /saf/arts-humanities-and-education

Text: Business and Entrepreneurship
URL: /saf/entrepreneurship

Text: Health and Medicine
URL: /saf/health-and-medicine

Text: Environment and Sustainability
URL: /saf/environment-and-sustainability

Text: Law, Journalism and Politics
URL: /saf/law-journalism-and-politics

Text: Local, National and Global Affairs

In [19]:
#will now keep only news article links
import re
from urllib.parse import urljoin

#creating a list
article_urls = []

for link in links:
    full_url = urljoin(url, link["href"])
    if full_url.startswith("https://news.asu.edu/20") or full_url.startswith("https://news.asu.edu/node/"):
        article_urls.append(full_url)
print("Total ## of article links:", len(article_urls))
print(article_urls)

Total ## of article links: 57
['https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election', 'https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election', 'https://news.asu.edu/20261109-sun-devil-community-accounting-what-was-lost', 'https://news.asu.edu/20261112-environment-and-sustainability-asu-professor-named-states-wildlife-veterinarian', 'https://news.asu.edu/20261106-health-and-medicine-testing-healing-power-dolphins-ptsd', 'https://news.asu.edu/20261106-sun-devil-community-helping-veterans-rebuild-health-confidence-and-connection', 'https://news.asu.edu/20261103-science-and-technology-asu-provide-energy-water-systems-support-us-air-force', 'https://news.asu.edu/20261104-university-news-how-arizona-building-futureready-public-workforce', 'https://news.asu.edu/20261102-university-news-classroom-career-asu-graduates-stand-out-employers', 'https://news.asu.edu/20260511-university-new

The next step is to get the keywords. Two ways that come to my mind are - either just by getting keywords from the link itself OR by opening the articles, reading the text and then finding a keyword.

But I am not sure how the first option would work. So for now I am going to go with the latter option. Another reasoning that makes me go with option 2 is because Dr. Halavais mentions in the assignment, “They seem to be using terms like ‘medicine,’ ‘breakthrough,’ and ‘upcoming,’ though there may be others.” This means the keywords are already set and urls alone cannot tell me, so I will have to produce python code that reads it for me and tells me if the keywords are used or not.

In [18]:
# article and reading its text
article_response = requests.get(article_urls[0])
article_soup = BeautifulSoup(article_response.text, "html.parser")
main_content = article_soup.find("main")
if main_content:
    article_text = main_content.get_text(" ", strip=True)
    print(article_text)

Local, national and global affairs 2 ASU students make history in Arizona election Armando Montero elected to Arizona Legislature at 25; Sebastian Matta wins Maricopa County Superior Court clerk race at 19 Photo by Chris Goulet/Arizona State University By Dolores Tropiano | November 10, 2026 Story highlights On Tuesday, ASU students Armando Montero and Sebastian Matta won their respective races in the Arizona midterms. Montero is among the first members of Gen Z to serve in the Arizona Legislature; at 19, Matta is the youngest person elected as the Maricopa County Superior Court clerk In January, when thousands of incoming public servants are ushered into office across the country, two Arizona State University students will be among them. In what is being considered historic wins, Sebastian Matta and Armando Montero won their respective races in the Arizona midterms, becoming among the first members of Generation Z to serve in Arizona state government. Matta, 19, a second-year student 

In [20]:
#searching articles with the keywords
keywords = ["medicine", "breakthrough", "upcoming"]

for word in keywords:
    if word in article_text.lower():
        print("Found:", word)
    else:
        print("Not found:", word)

Not found: medicine
Not found: breakthrough
Not found: upcoming


This means none of the words appeared in the articles.

Maybe instead of doing one article, I can search for all articles at once.

In [21]:
keywords = ["medicine", "breakthrough", "upcoming"]
results = []

for article_url in article_urls:
    response = requests.get(article_url)
    soup = BeautifulSoup(response.text, "html.parser")
    main_content = soup.find("main")

    if main_content:
        article_text = main_content.get_text(" ", strip=True).lower()

        for word in keywords:
            if word in article_text:
                results.append({"keyword": word, "url": article_url})
                print("Found:", word, article_url)

print("Total matches = ", len(results))

Found: breakthrough https://news.asu.edu/20261109-sun-devil-community-accounting-what-was-lost
Found: medicine https://news.asu.edu/20261112-environment-and-sustainability-asu-professor-named-states-wildlife-veterinarian
Found: medicine https://news.asu.edu/20261106-health-and-medicine-testing-healing-power-dolphins-ptsd
Found: medicine https://news.asu.edu/20261106-sun-devil-community-helping-veterans-rebuild-health-confidence-and-connection
Found: medicine https://news.asu.edu/20261106-health-and-medicine-testing-healing-power-dolphins-ptsd
Found: medicine https://news.asu.edu/20261113-health-and-medicine-extreme-heat-may-pose-hidden-risk-cognitive-health
Found: medicine https://news.asu.edu/20261112-health-and-medicine-you-arent-just-what-you-eat
Found: medicine https://news.asu.edu/20261106-health-and-medicine-testing-healing-power-dolphins-ptsd
Found: breakthrough https://news.asu.edu/20261103-science-and-technology-what-eye-may-reveal-about-alzheimers
Found: medicine https://news

so only 12/57 matches had the keyword. I guess i will just store that into the csv.

But, I think there could be others that might be interesting.

I am going to chose the following cz why not.
keywords = ["memory", "connection", "psychology", "health", "research"]

Oh, I also forgot to add sleep. I will add it in the next chunk.

In [36]:
import time

keywords = ["medicine", "breakthrough", "upcoming", "connection",
            "psychology", "health", "research"]

results = []

for article_url in article_urls:
    time.sleep(3)

    response = requests.get(article_url, timeout=30)
    soup = BeautifulSoup(response.text, "html.parser")
    main_content = soup.find("main")

    if main_content:
        article_text = main_content.get_text(" ", strip=True).lower()

        for word in keywords:
            if word in article_text:
                results.append({"keyword": word, "url": article_url})
                print("Found:", word, article_url)

print("Total matches =", len(results))

Found: health https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election
Found: research https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election
Found: health https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election
Found: research https://news.asu.edu/20261110-local-national-and-global-affairs-2-asu-students-make-history-arizona-election
Found: breakthrough https://news.asu.edu/20261109-sun-devil-community-accounting-what-was-lost
Found: connection https://news.asu.edu/20261109-sun-devil-community-accounting-what-was-lost
Found: research https://news.asu.edu/20261109-sun-devil-community-accounting-what-was-lost
Found: medicine https://news.asu.edu/20261112-environment-and-sustainability-asu-professor-named-states-wildlife-veterinarian
Found: research https://news.asu.edu/20261112-environment-and-sustainability-asu-professor-named-st

123 total matches - can definitely not mean 123 separate articles. I think the code counts each keyword found in each article separately. I looked up more closely and saw one article had “medicine,” “health,” and “research” adds 3 matches. This could be a reason of such a big size. for such a big size. I should check how many distinct keyword-URL pairs

In [38]:
df = pd.DataFrame(results)

print("Total matches = ", len(df))
print("Unique keyword–URL pairs = ", len(df.drop_duplicates()))
print("Unique article URLs with matches = ", df["url"].nunique())

df = df.drop_duplicates()
print(df)

Total matches =  123
Unique keyword–URL pairs =  67
Unique article URLs with matches =  31
          keyword                                                url
0          health  https://news.asu.edu/20261110-local-national-a...
1        research  https://news.asu.edu/20261110-local-national-a...
4    breakthrough  https://news.asu.edu/20261109-sun-devil-commun...
5      connection  https://news.asu.edu/20261109-sun-devil-commun...
6        research  https://news.asu.edu/20261109-sun-devil-commun...
..            ...                                                ...
103      research  https://news.asu.edu/20261030-university-news-...
104      research  https://news.asu.edu/20261028-university-news-...
105      medicine  https://news.asu.edu/20261113-university-news-...
106        health  https://news.asu.edu/20261113-university-news-...
107      research  https://news.asu.edu/20261113-university-news-...

[67 rows x 2 columns]


Ok, this makes sense. I will now put it into csv

In [37]:
import os
# Appending results to the CSV file.
filename = "asu_news_matches_mod4.csv"
df.to_csv(filename, mode="a", header=not os.path.exists(filename), index=False)
